In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import json
import glob
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
sys.path.insert(0, str(Path("../src").resolve()))
plt.ioff()
from model import GeosteeringCorrelationModel
from dataset import prepare_well_dataframe, load_typewell
from train_full import navigate_well  # SAME code path as validation

In [ ]:
# ---------------------------------------------------------------------
# 1. Load normalization stats + task config saved by train_full.py
# ---------------------------------------------------------------------
norm_stats_path = Path("../src/models/normalization_stats.json")
if not norm_stats_path.exists():
    raise FileNotFoundError(
        f"Normalization stats file missing: {norm_stats_path.resolve()}. "
        "Run train_full.py first -- it saves this file next to the checkpoint."
    )
with open(norm_stats_path) as f:
    norm_stats = json.load(f)

if norm_stats.get("target") != "typewell_registration_v2_per_signal_norm":
    raise ValueError(
        "normalization_stats.json is from an OLD pipeline version. "
        "Re-run train_full.py with the updated code first."
    )

WINDOW_SIZE = norm_stats["window_size"]
JITTER_FT = norm_stats["jitter_ft"]
TYPE_HALF_RANGE = norm_stats["type_half_range"]
TYPE_LEN = norm_stats["type_len"]
print(f"Loaded stats/config: {norm_stats}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference running on: {device}")

# ---------------------------------------------------------------------
# 2. Load best model
# ---------------------------------------------------------------------
model_path = Path("../src/models/best_geosteering_model.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Model file missing: {model_path.resolve()}")

model = GeosteeringCorrelationModel(type_len=TYPE_LEN)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.to(device)
model.eval()
print("Model successfully loaded.")

In [ ]:
# ---------------------------------------------------------------------
# 3. Navigation
# ---------------------------------------------------------------------
# The sequential navigation (re-anchoring at every known TVT_input point,
# dead reckoning + CLIPPED model re-registration in blind stretches,
# reverse dead reckoning for a leading gap) lives in train_full.navigate_well
# and is imported above -- inference and anchored validation share the
# exact same code path.
# Set to False to create a PURE DEAD-RECKONING submission (model
# corrections disabled) — useful once to calibrate the leaderboard
# metric and establish the baseline score.
USE_MODEL = True

cfg = {k: norm_stats[k] for k in
       ("window_size", "jitter_ft", "type_half_range", "type_len")}
print(f"Navigation config: {cfg}")
print(f"USE_MODEL = {USE_MODEL}")

In [ ]:
# ---------------------------------------------------------------------
# 4. Inference & Plot Loop
# ---------------------------------------------------------------------
test_dir = Path("../data/test")
test_files = list(test_dir.glob("*__horizontal_well.csv"))
print(f"Found test wells: {len(test_files)}\n")

submission_data = []
start_time = time.perf_counter()
os.makedirs("img", exist_ok=True)

for file_path in tqdm(test_files, desc="Navigate and plot wells", colour="green"):
    well_id = file_path.name.split("__")[0]
    type_path = test_dir / f"{well_id}__typewell.csv"
    if not type_path.exists():
        raise FileNotFoundError(
            f"Typewell missing for test well {well_id}: {type_path}")

    df = prepare_well_dataframe(pd.read_csv(file_path))
    df["MD"] = pd.to_numeric(df["MD"], errors="coerce")
    typewell = load_typewell(type_path)  # self-normalized (robust median/IQR)

    tvt_input = pd.to_numeric(df["TVT_input"], errors="coerce").to_numpy(np.float64)

    est = navigate_well(model if USE_MODEL else None, df, typewell,
                        norm_stats, cfg, device,
                        tvt_input, stride=1)
    df["TVT_pred"] = est

    missing_mask = df["TVT_input"].isna()

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(df["MD"], df["TVT_input"], color="green", linewidth=2,
            label="Known TVT_input")
    ax.plot(df["MD"], df["TVT_pred"], color="purple", linestyle="--",
            linewidth=2, label=("Model Navigation (TVT)" if USE_MODEL else "Dead Reckoning (TVT)"))
    if missing_mask.any():
        eval_start_md = float(df.loc[missing_mask.idxmax(), "MD"])
        ax.axvline(x=eval_start_md, color="red", linestyle="-", alpha=0.5,
                   label="Start blind flight (NaN)")
        ax.axvspan(eval_start_md, float(df["MD"].max()), color="red", alpha=0.1)
    ax.set_title(f"Geosteering Navigation for Well {well_id}")
    ax.set_xlabel("Measured Depth (MD)")
    ax.set_ylabel("True Vertical Thickness (TVT)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"img/inference_plot_{well_id}_registration1.png", dpi=100,
                bbox_inches="tight")
    plt.close(fig)

    # --- Kaggle Submission ---
    fallback = df["TVT_input"].ffill().bfill()
    for idx in df[missing_mask].index:
        tvt_value = df.loc[idx, "TVT_pred"]
        if pd.isna(tvt_value):
            tvt_value = fallback.loc[idx]
        submission_data.append({"id": f"{well_id}_{idx}", "tvt": tvt_value})

In [ ]:
# ---------------------------------------------------------------------
# 5. Create and save Submission CSV
# ---------------------------------------------------------------------
submission_df = pd.DataFrame(submission_data)

# index=False is extremely important for Kaggle, otherwise an extra column is generated
submission_df.to_csv("submission.csv", index=False)

total_time = time.perf_counter() - start_time
print(f"\nDone! The file 'submission.csv' was successfully created in {total_time:.1f} seconds.")
print(f"Number of missing values predicted for Kaggle: {len(submission_df)}")